# Tasks 4 and 5 — Frozen CLIP Stage A
Run this after notebook 00 has prepared the same Colab runtime and Task 2 has passed. This notebook never opens `final_test`.

In [ ]:
from pathlib import Path
import shutil
import subprocess

PROJECT_ROOT = Path('/content/cya-techjam26')
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
TASK2_ROOT = ARTIFACT_ROOT / 'task2'
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts')

assert PROJECT_ROOT.is_dir(), 'Run notebook 00 first'
assert Path('/content/hackathon_data/raw/sid_set/images').is_dir(), 'Stage SID first'
subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)
subprocess.run(['make', 'smoke'], cwd=PROJECT_ROOT, check=True)

Matched pilot JPEGs are disposable. Regenerate them only if the current runtime does not contain both complete 2,000-image candidates.

In [ ]:
DRIVE_TASK2_ROOT = DRIVE_ARTIFACT_ROOT / 'task2'
TASK2_ROOT.mkdir(parents=True, exist_ok=True)
if DRIVE_TASK2_ROOT.is_dir():
    for path in DRIVE_TASK2_ROOT.iterdir():
        if path.is_file() and path.suffix in {'.csv', '.json'} and not (TASK2_ROOT / path.name).exists():
            shutil.copy2(path, TASK2_ROOT / path.name)

manifests = [TASK2_ROOT / 'fixed_q96_manifest.csv', TASK2_ROOT / 'uniform_q95_q100_manifest.csv']
candidate_root = TASK2_ROOT / 'matched_candidates'
candidate_counts = {policy: sum(1 for path in (candidate_root / policy).rglob('*.jpg')) for policy in ('fixed_q96', 'uniform_q95_q100')}

if not all(path.is_file() for path in manifests) or set(candidate_counts.values()) != {2000}:
    print(f'Regenerating Task 2 pilots; found {candidate_counts}')
    subprocess.run(['make', 'task2-pilots', f'ARTIFACT_ROOT={ARTIFACT_ROOT}'], cwd=PROJECT_ROOT, check=True)
else:
    print('Task 2 pilot images are ready')

Run both matching policies over three seeds. The first run downloads CLIP and populates the shared embedding cache. Start with batch size 8 and lower it after a CUDA out-of-memory error.

In [ ]:
policies = ('fixed_q96', 'uniform_q95_q100')
seeds = (42, 43, 44)
physical_batch_size = 8

for policy in policies:
    manifest = TASK2_ROOT / f'{policy}_manifest.csv'
    for seed in seeds:
        print(f'\nRunning {policy}, seed {seed}')
        subprocess.run([
            'python', 'scripts/train_clip_baseline.py',
            '--manifest', str(manifest),
            '--matching-policy', policy,
            '--output-root', str(ARTIFACT_ROOT / 'task4'),
            '--cache-root', '/content/clip_embedding_cache',
            '--seed', str(seed),
            '--physical-batch-size', str(physical_batch_size),
        ], cwd=PROJECT_ROOT, check=True)

Generate clean-selection reports. `selection_score` remains null until Task 3 adds robustness cells; this is intentional.

In [ ]:
for policy in policies:
    for seed in seeds:
        predictions = ARTIFACT_ROOT / 'task4' / policy / f'seed_{seed}' / 'best_clean_predictions.csv'
        output = ARTIFACT_ROOT / 'task5' / policy / f'seed_{seed}'
        subprocess.run([
            'python', 'scripts/evaluate_predictions.py',
            '--predictions', str(predictions),
            '--output', str(output),
        ], cwd=PROJECT_ROOT, check=True)

Compare both matching policies across all three clean Stage A seeds. This provisional clean-only choice must be revisited with the locked 50/50 score after Task 3.

In [ ]:
subprocess.run(['make', 'task4-compare', f'ARTIFACT_ROOT={ARTIFACT_ROOT}'], cwd=PROJECT_ROOT, check=True)

In [ ]:
for task in ('task4', 'task5'):
    source = ARTIFACT_ROOT / task
    destination = DRIVE_ARTIFACT_ROOT / task
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print(f'Copied {source} to {destination}')